## Variables explicatives de cropcut et saison mesurée en Éthiopie

`cropcut` réunit les rendements mesurés par coupe dans les grappes de l'enquête éthiopienne. Il sert de validation externe et n'est jamais utilisé pour l'entraînement, selon `docs/evaluation_protocol.md`. Pour évaluer un modèle sur ces mesures, chacune doit recevoir les variables explicatives qu'aurait une parcelle de `survey` de même grappe, de même culture et de même vague. Ce notebook contrôle ces variables et exploite les dates de coupe, absentes de `survey`, pour établir la saison mesurée par l'enquête éthiopienne.

**Règles reprises de `survey`**

- Le lieu d'une mesure est le voisinage de 5 km de sa grappe, ou sa zone de repli lorsque les coordonnées manquent.
- Le calendrier est celui de `HarvestStat` pour la zone et la culture, selon l'ordre de priorité établi dans `06_climate_variables.ipynb`.
- La vague dont l'année `Year` vaut A + 1 commence en A, comme la campagne `AgYearStart` de `survey`. La récolte est placée l'année A + 1 lorsque le calendrier chevauche deux années civiles.

**Préalable**

La préparation des données doit avoir été relancée par `python -m src.data.prepare`, qui ajoute l'identifiant `id_mesure`. Les variables sont construites par `python -m src.features.variables_cropcut`, qui écrit `climat_cropcut.csv.gz`, `sols_cropcut.csv.gz` et `relief_cropcut.csv.gz` dans `data/processed`. Le notebook s'exécute en deux minutes environ.

In [1]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.data.load import DOSSIER_TRAITE
from src.features.calendrier import attribuer_calendrier, calendriers_harveststat
from src.features.climat import DOSSIER_MAGASIN, ouvrir_magasin
from src.features.controles import alignement_survey, cumuls_par_lieu, valeur_fenetres
from src.features.geographie import zone_survey
from src.features.variables_cropcut import cropcut_comme_survey, date_calendrier_ethiopien

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
cropcut = pd.read_csv(DOSSIER_TRAITE / "cropcut_validation.csv.gz", low_memory=False)
survey = pd.read_csv(DOSSIER_TRAITE / "survey_modelisation.csv.gz", low_memory=False)
survey_ethiopie = survey[survey["Country"] == "Ethiopia"].copy()

### 1. Tests unitaires

Les tests propres à cette étape vérifient l'identifiant des mesures, la conversion des dates du calendrier éthiopien sur le nouvel an et la fête de Meskel, et l'alignement des vagues de `cropcut` sur celles de `survey`.

In [2]:
execution = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=RACINE, capture_output=True, text=True,
)
print(execution.stderr.strip().splitlines()[-3:])

['Ran 90 tests in 1.374s', '', 'OK']


### 2. Identifiant des mesures

Une mesure doit pouvoir être rattachée sans ambiguïté à ses variables explicatives et à sa partition d'évaluation.

In [3]:
cles = ["holder_id", "parcel_id", "field_id", "Crop", "Year"]
print("Mesures :", len(cropcut), "| identifiants uniques :", cropcut["id_mesure"].is_unique,
      "| doublons de la clé d'origine :", int(cropcut.duplicated(cles).sum()),
      "| mesures sans identifiant d'exploitant :", int(cropcut["holder_id"].isna().sum()))
display(cropcut.groupby("Source").agg(mesures=("id_mesure", "size"), sans_exploitant=("holder_id", lambda s: int(s.isna().sum())),
                                      annee=("Year", "first")))
print("Vagues de survey en Éthiopie :", survey_ethiopie.groupby("Source")["AgYearStart"].first().to_dict())

Mesures : 11263 | identifiants uniques : True | doublons de la clé d'origine : 2502 | mesures sans identifiant d'exploitant : 2904


,mesures,sans_exploitant,annee
Source,,,
ETH2011,1768,0,2012
ETH2013b,2695,0,2014
ETH2015,3896,0,2016
ETH2018,2904,2904,2019


Vagues de survey en Éthiopie : {'LSMS_ETH2011': 2011, 'LSMS_ETH2013b': 2013, 'LSMS_ETH2015': 2015, 'LSMS_ETH2018': 2018, 'LSMS_ETH2021': 2021}


**Constat**

- La combinaison de l'exploitant, de la parcelle, du champ, de la culture et de l'année ne distingue pas les mesures, avec 2 502 doublons. L'identifiant d'exploitant manque pour les 2 904 mesures de la vague de 2018.
- L'année `Year` de chaque vague de `cropcut` suit d'un an l'année `AgYearStart` de la vague correspondante de `survey`.

**Décision**

`id_mesure` est formé de la source et de la position de la mesure dans le fichier brut, selon la convention d'`id_ligne` dans `survey`. Il est ajouté par la fonction `identifier_mesures` de `src/data/cropcut.py`. La préparation relancée laisse inchangés les fichiers de `regional` et de `survey`, dont le contenu est identique octet pour octet.

### 3. Dates de coupe et saison mesurée

Les dates de coupe sont un fait d'observation propre à `cropcut`. La préparation les avait interprétées dans le calendrier éthiopien. Ce contrôle vérifie cette interprétation, puis convertit les dates en dates grégoriennes en supposant que la coupe a lieu au cours de l'année éthiopienne qui commence en septembre de l'année de début de campagne.

In [4]:
datees = cropcut[cropcut["cropCutMonth"].notna()]
print("Mesures datées :", len(datees), "| sans date :", int(cropcut["cropCutMonth"].isna().sum()),
      "| dates impossibles même en calendrier éthiopien :", int(cropcut["flag_date_mesure_invalide"].sum()))
print("Mesures du 30 du deuxième mois :", int(((datees["cropCutMonth"] == 2) & (datees["cropCutDay"] == 30)).sum()),
      "| mesures d'un 31 :", int((datees["cropCutDay"] == 31).sum()),
      "| mesures du treizième mois :", int((datees["cropCutMonth"] == 13).sum()))

valides = datees[~datees["flag_date_mesure_invalide"]].copy()
valides["date"] = date_calendrier_ethiopien(valides["Year"] - 1, valides["cropCutMonth"], valides["cropCutDay"].fillna(15)).to_numpy()
valides["mois"] = valides["date"].dt.to_period("M")
display(pd.crosstab(valides["mois"], valides["Source"]))
debut_campagne = valides["Year"] - 1
print("Part des coupes d'octobre à décembre de l'année de début de campagne :",
      round((valides["date"].dt.month.isin([10, 11, 12]) & (valides["date"].dt.year == debut_campagne)).mean(), 3))
print("Part des coupes antérieures au 1er janvier de l'année Year :",
      round((valides["date"] < pd.to_datetime(valides["Year"].astype(str) + "-01-01")).mean(), 3))
print("Mois de coupe le plus fréquent par culture :", valides.groupby("Crop")["date"].apply(lambda d: int(d.dt.month.mode()[0])).to_dict())

Mesures datées : 8566 | sans date : 2697 | dates impossibles même en calendrier éthiopien : 18
Mesures du 30 du deuxième mois : 116 | mesures d'un 31 : 0 | mesures du treizième mois : 2


Source,ETH2011,ETH2015,ETH2018
mois,,,
2011-09,34,0,0
2011-10,590,0,0
2011-11,794,0,0
2011-12,296,0,0
2012-01,46,0,0
2012-05,1,0,0
2012-06,1,0,0
2015-09,0,51,0
2015-10,0,905,0


Part des coupes d'octobre à décembre de l'année de début de campagne : 0.905
Part des coupes antérieures au 1er janvier de l'année Year : 0.921
Mois de coupe le plus fréquent par culture : {'Barley': 10, 'Beans': 11, 'Groundnut': 11, 'Maize': 10, 'Millet': 11, 'Rice': 11, 'Sesame': 11, 'Sorghum': 11, 'Soybean': 11, 'Sweet potato': 10, 'Teff': 11, 'Wheat': 11}


**Constat**

- 8 566 mesures sont datées. Les 2 697 mesures sans date comprennent toute la vague de 2013, et 18 dates sont impossibles même dans le calendrier éthiopien.
- 116 mesures sont datées du 30 du deuxième mois, date qui n'existe pas dans le calendrier grégorien, et aucune n'est datée d'un 31, alors que sept mois grégoriens en comptent un. Deux mesures tombent au treizième mois.
- Une fois converties, 90,5 % des coupes ont lieu d'octobre à décembre de l'année de début de campagne, et 92,1 % avant le 1er janvier de l'année `Year`. Le mois de coupe le plus fréquent est octobre ou novembre pour chaque culture.

**Analyse**

- Les dates suivent sans ambiguïté le calendrier éthiopien.
- Les coupes ont lieu pendant la récolte de la grande saison, appelée meher, entre le passage de l'enquête après les semis et son passage après la récolte. Les champs mesurés sont ceux que l'enquête a recensés après les semis de l'année de début de campagne. La saison mesurée est donc celle qui est semée cette année-là et récoltée à la fin de la même année civile, et non celle de l'année précédente.
- Le calendrier `HarvestStat` de l'Éthiopie place la récolte en janvier, alors que les coupes ont surtout lieu en octobre et en novembre. La fenêtre de saison se prolonge donc d'un à deux mois après la récolte réelle.

**Conséquences pour la suite**

La règle retenue pour `survey` en Éthiopie, qui place la saison de mai de l'année de début de campagne à janvier de l'année suivante, est confirmée par l'observation directe.

### 4. Test climatique sur les rendements mesurés

Le notebook `06_climate_variables.ipynb` avait trouvé que la pluie de la saison de l'année précédente expliquait mieux les rendements éthiopiens de `survey`, avec une corrélation de 0,111 contre -0,016. Le même test est appliqué ici aux rendements mesurés de `cropcut`, issus des mêmes grappes mais obtenus par pesée. Les deux hypothèses sont nommées comme dans le notebook 06, la récolte en `AgYearStart` désignant la saison semée l'année précédente.

In [5]:
regional = pd.read_csv(DOSSIER_TRAITE / "regional_modelisation.csv.gz", low_memory=False)
tables = calendriers_harveststat(regional)
dates = ouvrir_magasin("power")[1]
poids = pd.read_csv(DOSSIER_MAGASIN / "poids_survey.csv")
poids = poids[poids["lieu"].str.startswith("zone:")].assign(lieu=lambda d: d["lieu"].str[5:])
pluie, _, debut_imerg = cumuls_par_lieu(poids, "imerg", "IMERG_PRECTOT")


def cellules_de(df, cible, debut_campagne):
    df = df.assign(zone=zone_survey(df), crop=df["Crop"].str.strip(), AgYearStart=debut_campagne)
    calendrier = attribuer_calendrier(df, tables, {"zone": "zone", "niveau_1": "L1_GID", "pays": "L0_GID"}, "crop",
                                      [np.nan] * len(df), [np.nan] * len(df))
    cellules = (
        df.assign(ly=np.log(df[cible]), mois_semis=calendrier["mois_semis"], mois_recolte=calendrier["mois_recolte"])
        .dropna(subset=["zone", "mois_semis"])
        .groupby(["Country", "zone", "crop", "AgYearStart", "mois_semis", "mois_recolte"])
        .agg(ly=("ly", "mean"), parcelles=("ly", "size")).reset_index()
    )
    return cellules[cellules["parcelles"] >= 10]


hypotheses = {"récolte en AgYearStart": 0, "récolte en AgYearEnd": 1}
resultats = {}
for nom, cellules in [
    ("survey, rendement déclaré", cellules_de(survey_ethiopie, "YieldUsingGPS_ton_ha_", survey_ethiopie["AgYearStart"])),
    ("cropcut, rendement mesuré", cellules_de(cropcut, "Yield_ton_ha_", cropcut["Year"] - 1)),
]:
    valeurs = {h: valeur_fenetres(pluie, dates, cellules["zone"], cellules["AgYearStart"] + d, cellules["mois_semis"],
                                  cellules["mois_recolte"], premier_jour=debut_imerg, moyenne=False)
               for h, d in hypotheses.items()}
    resultats[nom] = alignement_survey(cellules, valeurs, transformation=np.log1p).iloc[0]
    print(nom, ": calendriers", cellules.groupby(["mois_semis", "mois_recolte"]).size().to_dict())
display(pd.DataFrame(resultats).T.round(3))

survey, rendement déclaré : calendriers {(5.0, 1.0): 773}
cropcut, rendement mesuré : calendriers {(5.0, 1.0): 421}


,récolte en AgYearStart,cellules récolte en AgYearStart,récolte en AgYearEnd,cellules récolte en AgYearEnd
"survey, rendement déclaré",0.111,605.0,-0.016,605.0
"cropcut, rendement mesuré",0.011,319.0,-0.010,319.0


**Constat**

- Sur 605 cellules de `survey`, la corrélation vaut 0,111 pour la saison de l'année précédente et -0,016 pour la saison de la règle, comme dans le notebook 06.
- Sur 319 cellules de `cropcut`, elle vaut 0,011 et -0,010. Aucune des deux hypothèses ne se distingue.

**Analyse**

- Le signal observé dans `survey` n'est pas reproduit par des rendements mesurés dans les mêmes grappes. Il est vraisemblablement fortuit, ce que rend possible le faible nombre de vagues.
- Le test climatique est peu puissant en Éthiopie, qui ne compte que quatre ou cinq vagues, et la pluie y explique faiblement le rendement, comme le montrent les corrélations proches de zéro de `cropcut`. L'observation directe des dates de coupe prime donc sur lui.

**Décision**

Le conflit éthiopien consigné dans `docs/explanatory_variables.md` est levé en faveur de la règle. L'analyse de sensibilité prévue lors de la modélisation n'est plus nécessaire et pourra être conservée comme simple contrôle.

### 5. Variables de cropcut

Les variables sont contrôlées sur leur couverture et sur leur cohérence avec `survey`. Une mesure et une parcelle de même grappe, de même culture et de même vague doivent recevoir exactement les mêmes valeurs.

In [6]:
climat = pd.read_csv(DOSSIER_TRAITE / "climat_cropcut.csv.gz", parse_dates=["debut", "fin"])
sols = pd.read_csv(DOSSIER_TRAITE / "sols_cropcut.csv.gz")
relief = pd.read_csv(DOSSIER_TRAITE / "relief_cropcut.csv.gz")
print("Lignes :", len(climat), len(sols), len(relief), "| même ordre que cropcut_validation :",
      all((t["id_mesure"].to_numpy() == cropcut["id_mesure"].to_numpy()).all() for t in [climat, sols, relief]))
print("Lieux :", climat["nature_lieu"].value_counts(dropna=False).to_dict())
print("Sources du calendrier :", climat["source_calendrier"].value_counts(dropna=False).to_dict())
display(climat.groupby(climat["id_mesure"].str.split("_").str[0])[["debut", "fin"]].agg(["min", "max"]))
print("Part des mesures avec tous les indicateurs climatiques :", round(climat.iloc[:, 9:].notna().all(axis=1).mean(), 4),
      "| avec sol :", round(sols["argile_0_30"].notna().mean(), 4), "| avec relief :", round(relief["altitude_moyenne"].notna().mean(), 4))

colonnes = ["pluie_cumul", "tmax_moyenne", "et0_cumul", "sequence_seche_max", "ph_0_30", "pente_moyenne"]
mesures = (climat.merge(cropcut[["id_mesure", "Crop", "Year"]], on="id_mesure")
           .merge(sols[["id_mesure", "ph_0_30"]], on="id_mesure").merge(relief[["id_mesure", "pente_moyenne"]], on="id_mesure"))
parcelles = (pd.read_csv(DOSSIER_TRAITE / "climat_survey.csv.gz")
             .merge(survey_ethiopie[["id_ligne", "Crop", "AgYearStart"]], on="id_ligne")
             .merge(pd.read_csv(DOSSIER_TRAITE / "sols_survey.csv.gz")[["id_ligne", "ph_0_30"]], on="id_ligne")
             .merge(pd.read_csv(DOSSIER_TRAITE / "relief_survey.csv.gz")[["id_ligne", "pente_moyenne"]], on="id_ligne")
             .assign(Year=lambda d: d["AgYearStart"] + 1, Crop=lambda d: d["Crop"].str.strip())
             .drop_duplicates(["lieu", "Crop", "Year"]))
appariees = mesures.merge(parcelles[["lieu", "Crop", "Year"] + colonnes], on=["lieu", "Crop", "Year"], suffixes=("", "_survey"))
print("Mesures appariées à une parcelle de survey de même grappe, culture et vague :", len(appariees))
display(pd.Series({c: np.isclose(appariees[c], appariees[c + "_survey"], equal_nan=True).mean() for c in colonnes},
                  name="part de valeurs identiques").to_frame().T)

Lignes : 11263 11263 11263 | même ordre que cropcut_validation : True
Lieux : {'grappe': 11181, nan: 82}
Sources du calendrier : {'zone': 10531, 'niveau_1': 596, 'pays': 136}


debut                   fin           
                 min        max        min        max
id_mesure                                            
ETH2011   2011-05-01 2011-05-01 2012-01-31 2012-01-31
ETH2013b  2013-05-01 2013-05-01 2014-01-31 2014-01-31
ETH2015   2015-05-01 2015-05-01 2016-01-31 2016-01-31
ETH2018   2018-05-01 2018-05-01 2019-01-31 2019-01-31

Part des mesures avec tous les indicateurs climatiques : 0.9927 | avec sol : 0.9927 | avec relief : 0.9927


Mesures appariées à une parcelle de survey de même grappe, culture et vague : 8235


,pluie_cumul,tmax_moyenne,et0_cumul,sequence_seche_max,ph_0_30,pente_moyenne
part de valeurs identiques,1.0,1.0,1.0,1.0,1.0,1.0


**Constat**

- Les trois fichiers comptent 11 263 lignes, dans l'ordre de `cropcut_validation.csv.gz`. 99,27 % des mesures reçoivent tous les indicateurs climatiques, les variables de sol et le relief. Les 82 autres n'ont ni coordonnées ni zone.
- Le calendrier provient de la zone pour 10 531 mesures, de la région pour 596 et du pays pour 136. Aucune mesure ne dépend de GGCMI.
- Toutes les fenêtres vont du 1er mai de l'année de début de campagne au 31 janvier de l'année suivante.
- Les 8 235 mesures appariées à une parcelle de `survey` de même grappe, culture et vague ont exactement les mêmes valeurs de pluie, de température, d'évapotranspiration, de séquence sèche, de pH et de pente.

**Analyse**

Les variables de `cropcut` sont construites par le même code que celles de `survey`, ce que confirme l'identité des valeurs. Un modèle entraîné sur `survey` peut donc être évalué sur `cropcut` sans écart de construction des variables.

### 6. Synthèse

**Résultats**

| Élément | Résultat |
|---|---|
| Identifiant | `id_mesure`, unique sur les 11 263 mesures de validation |
| Calendrier des dates de coupe | Éthiopien, 90,5 % des coupes d'octobre à décembre de l'année de début de campagne |
| Alignement de l'Éthiopie | Règle confirmée, conflit levé |
| Couverture des variables | 99,27 % des mesures |
| Cohérence avec `survey` | Valeurs identiques pour les 8 235 mesures appariées |

**Limites**

- La vague de 2013 n'a aucune date de coupe, et son alignement repose sur celui des autres vagues.
- La fenêtre de saison éthiopienne s'achève en janvier, un à deux mois après la plupart des coupes. Les indicateurs comptent donc quelques semaines sèches postérieures à la récolte, ce qui affecte surtout l'évapotranspiration cumulée et le bilan hydrique.
- L'argument fondé sur les dates suppose que la coupe a lieu entre les deux passages de l'enquête, conformément au déroulement publié de l'enquête éthiopienne.